# TorchTitan-NPU 核心特性

TorchTitan 提供模块化的大模型训练栈，TorchTitan-NPU 在此基础上适配 Ascend NPU 模型、算子和执行环境。本章先看 Wordle 训练实际用到的 FSDP2、内存管理和 TND 变长注意力，再介绍上下文并行如何扩展长序列训练。

---


## 前置要求

为了充分掌握本章内容，你应已具备以下能力：

- 已完成第 5 章学习，理解训练后端切换的范围和调用链。
- 了解 PyTorch 分布式训练中的 rank、参数分片和集合通信。
- 理解强化学习样本长度动态变化以及 packed batch 的基本含义。

---

## 章节目标

完成本章后，你将能够：

- 说明 DeviceMesh 如何组织两卡 FSDP2。
- 理解 offload、reshard、混合精度和激活重计算的职责。
- 解释 Qwen3 TND 变长注意力中的 T、N、D 与序列边界元数据。
- 说明 TND 为什么适合长度动态变化的强化学习样本。
- 理解上下文并行适用的长序列场景及其通信代价。
- 说明分布式训练状态如何支持 Actor 到 vLLM 的权重同步。

---

## 本章内容

- [6.1 章节介绍](06.01_chapter_intro.ipynb)
- [6.2 FSDP2 与可组合并行](06.02_fsdp2_and_parallelism.ipynb)
- [6.3 Wordle 训练使用的 TorchTitan-NPU 特性](06.03_features_used_in_wordle.ipynb)
- [6.4 章节练习](06.04_chapter_practice.ipynb)


---

## 本章使用的配置

| 类别 | Wordle 训练配置 |
| --- | --- |
| 两卡并行 | DP shard 为 2、CP 为 1，使用 FSDP2 |
| 参数与优化器卸载 | Actor 卸载参数与优化器状态；Ref 仅卸载参数 |
| forward 后重新分片 | Actor 使用 `reshard_after_forward=always`；Ref 保持 forward-only 默认策略 |
| 训练精度与激活重计算 | TorchTitan 默认使用 BF16 参数、FP32 reduction，并启用激活重计算 |
| Qwen3 NPU 计算路径 | 启用 NPU converter 与 TND 变长注意力 |
| CP、TP、PP、EP | 介绍其组合方式，三步训练中不启用 |
| `torch.compile` | 保持关闭 |

Ref 只计算冻结模型的前向和 log-prob，不创建优化器，所以只需卸载参数。Actor 需要执行反向传播和参数更新，还要管理优化器状态，两者的配置也因此不同。

这套配置用 FSDP2 在两张 NPU 之间分片训练状态，并用 TND 减少 packed 变长样本中的 padding 计算。上下文并行可以进一步分摊长序列的激活和注意力计算，但也会增加通信；第 6.2 节会结合当前两卡配置说明何时值得启用。
